<a href="https://colab.research.google.com/github/ShubhendraP/AgenticAI2026/blob/weekly-classes/pharma_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this cell first to install / upgrade LangChain (only needed once per Colab session)
!pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24 --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 27.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.


In [ ]:
# ── Google Colab API Key Setup ──────────────────────────────────────────────
# Store your OPENAI_API_KEY in Colab Secrets (key icon in left sidebar).
# This replaces the local .env file approach used in a local environment.
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_BASE_URL'] = 'https://openai.vocareum.com/v1'


# LangChain Tool-Calling Agent — Pharmaceutical Assistant

## Overview
This notebook builds a **Pharmaceutical AI Agent** that can answer complex drug-related queries
by orchestrating multiple specialised tools:
- Drug information lookup from an internal mock database.
- Contraindication / safety checks.
- Weight-based dosage calculation.
- Simulated external medical literature search (PubMed-style API).

### Why Use an Agent Instead of a Simple LLM Call?
A plain LLM can hallucinate drug details or dosage figures. By **grounding the LLM with tools**,
we force it to retrieve factual data from controlled sources before answering,
significantly improving accuracy and safety in high-stakes domains like healthcare.

In [ ]:
# check langchain version it should be 0.3 if not uncomment below pip command to install the correct version
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.3.27
0.3.24


In [ ]:
# Pharma Agent

In [ ]:
# See pharma_agent.md for a full description of this notebook's purpose and design.

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent

# API key already loaded from Colab Secrets in the setup cell above.

# temperature=0 → deterministic outputs; important for medical/safety-critical applications
# where reproducibility matters more than creativity.
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Simulated Drug Database (Mock Data)

In a real production system this dictionary would be replaced by a call to a
hospital formulary database, an EHR system, or a drug information API (e.g. OpenFDA, RxNorm).
For learning purposes, a **Python dict** is used as a stand-in (`mock_database`).

Each entry stores:
- **`use`** — the primary clinical indication for the drug.
- **`contraindications`** — conditions or situations where the drug should not be prescribed.

In [ ]:
mock_database = {
    "aspirin": {"use": "Pain relief", "contraindications": "Ulcers, bleeding disorders"},
    "metformin": {"use": "Diabetes Type 2", "contraindications": "Severe kidney issues"},
    "ibuprofen": {"use": "Inflammation and fever", "contraindications": "Asthma, bleeding disorders"},
    "paracetamol": {"use": "Pain relief and fever", "contraindications": "Liver disease"},
    "amoxicillin": {"use": "Bacterial infections", "contraindications": "Penicillin allergy"},
    "atorvastatin": {"use": "Cholesterol reduction", "contraindications": "Liver disease"},
    "lisinopril": {"use": "Hypertension", "contraindications": "Pregnancy, angioedema history"},
    "omeprazole": {"use": "Acid reflux", "contraindications": "Liver disease"},
    "simvastatin": {"use": "Cholesterol reduction", "contraindications": "Liver disease"},
    "levothyroxine": {"use": "Hypothyroidism", "contraindications": "Untreated adrenal insufficiency"},
    "ciprofloxacin": {"use": "Bacterial infections", "contraindications": "Tendon disorders"},
    "clopidogrel": {"use": "Prevent blood clots", "contraindications": "Active bleeding"},
    "fluoxetine": {"use": "Depression", "contraindications": "MAOI use"},
    "warfarin": {"use": "Prevent blood clots", "contraindications": "Bleeding disorders"}
}

## Designing Domain-Specific Tools

Well-designed tools follow three principles:
1. **Single responsibility** — each tool does one clearly defined job (lookup, safety check, calculation, search).
2. **Descriptive docstrings** — the LLM relies on the docstring to decide *when* and *how* to use the tool;
   vague docstrings lead to wrong tool selection.
3. **Typed signatures** — type hints (`str`, `float`, `int`) help the LLM pass arguments in the correct format.

Four tools are defined:
| Tool | Purpose |
|---|---|
| `lookup_drug` | Retrieves drug information from the mock database |
| `safety_check` | Returns contraindications for a drug |
| `dosage_calculator` | Computes weight-based dosage from `patient_weight` and `mg_per_kg` |
| `external_medical_search` | Simulates a PubMed/clinical API call for literature summaries |

In [ ]:
# ── Tool 1: Drug Information Lookup ────────────────────────────────────────────
# Retrieves use-case and contraindication data from the mock database.
# In production: replace mock_database lookup with a real drug API or database query.
@tool
def lookup_drug(drug_name: str) -> str:
    """this would look up drug information from a database"""
    drug = mock_database.get(drug_name.lower())
    return drug if drug else "Drug not found."

# ── Tool 2: Safety / Contraindication Check ─────────────────────────────────────
# Explicitly surfaces contraindications — the LLM calls this before recommending any drug.
@tool
def safety_check(drug_name: str) -> str:
    """this would check for contraindications"""
    drug = mock_database.get(drug_name.lower())
    if not drug:
        return "No safety data available."
    return f"Contraindications: {drug['contraindications']}"

# ── Tool 3: Weight-Based Dosage Calculator ───────────────────────────────────────
# Computes dose = patient_weight (kg) × mg_per_kg.
# Input format: 'patient_weight=75 mg_per_kg=5' (space-separated key=value pairs).
@tool
def dosage_calculator(input_data: str) -> str:
    """
    Example: 'patient_weight=75 mg_per_kg=5'
    """
    values = dict(item.split("=") for item in input_data.split())
    dose = float(values["patient_weight"]) * float(values["mg_per_kg"])
    return f"Recommended dose: {dose} mg"

# ── Tool 4: External Medical Literature Search ───────────────────────────────────
# Simulates a call to PubMed / a clinical trials API.
# In production: replace the return statement with a real HTTP request to the API.
@tool
def external_medical_search(query: str) -> str:
    """this would call an external medical database API"""
    return f"(Simulated summary from PubMed/clinical API) About: {query}"
    # In real implementation, integrate with clinical trial APIs


# Bundle all tools into a list that will be passed to the agent.
tools = [lookup_drug, safety_check, dosage_calculator, external_medical_search]

## Building and Running the Pharmaceutical Agent

The agent is assembled from three parts:
- **Prompt** — instructs the LLM to behave as an expert pharmaceutical assistant and to use tools accurately.
- **`create_tool_calling_agent`** — compiles the LLM + tools + prompt into an agent object.
- **`AgentExecutor`** — runs the think-act-observe loop.

Note: `max_iterations=1` is set to limit cost in demos; increase it for full multi-step reasoning.

In [ ]:
# Agent Prompt Template
from langchain_core.prompts import ChatPromptTemplate

# The prompt defines the agent's persona and the required placeholders:
# - {input}            → the user's question at runtime
# - {agent_scratchpad} → the agent's working memory (tool calls & observations)
prompt = ChatPromptTemplate.from_template("""
You are an expert pharmaceutical assistant. Use the tools provided to answer pharmaceutical-related queries accurately.

User Request: {input},
("placeholder", "{agent_scratchpad}"
""")

# Compile: LLM + tools + prompt → agent object (no execution yet)
agent = create_tool_calling_agent(llm=model, tools=tools, prompt=prompt)

# AgentExecutor runs the think-act-observe loop.
# max_iterations=1 limits tool-calling rounds to 1 to keep demo costs low;
# increase this for complex multi-step queries.
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run a complex multi-tool query to demonstrate the agent's reasoning.
output = executor.invoke({"input": "Provide information on ibuprofen, check its safety for a patient with asthma, calculate dosage for a 70kg patient at 10mg/kg, "})
print("Agent Output:", output)



> Entering new AgentExecutor chain...

Invoking: `lookup_drug` with `{'drug_name': 'ibuprofen'}`


{'use': 'Inflammation and fever', 'contraindications': 'Asthma, bleeding disorders'}
Invoking: `safety_check` with `{'drug_name': 'ibuprofen'}`


Contraindications: Asthma, bleeding disorders
Invoking: `dosage_calculator` with `{'input_data': 'patient_weight=70 mg_per_kg=10'}`


Recommended dose: 700.0 mg
Invoking: `lookup_drug` with `{'drug_name': 'ibuprofen'}`


{'use': 'Inflammation and fever', 'contraindications': 'Asthma, bleeding disorders'}
Invoking: `safety_check` with `{'drug_name': 'ibuprofen'}`


Contraindications: Asthma, bleeding disorders
Invoking: `dosage_calculator` with `{'input_data': 'patient_weight=70 mg_per_kg=10'}`


Recommended dose: 700.0 mg### Ibuprofen Information
- **Uses**: Ibuprofen is commonly used for the treatment of inflammation and fever.
- **Contraindications**: It is contraindicated in patients with asthma and bleeding disorders.

### Safety Check for

In [ ]:
# sample test query
# "Provide information on ibuprofen, check its safety for a patient with asthma, calculate dosage for a 70kg patient at 10mg/kg, and summarize recent studies on its efficacy."
# "What are the uses and contraindications of metformin?"
# "What is the medicine for pain relief and what are its contraindications?"
# "What medicine recommend for a patient with diabetes type 2 who has kidney issues?"
# "For fever management, what drug would you suggest and what safety checks should be considered?"